# Bazalt in a notebook

Bazalt needs no window and no display. It starts the window system only when you ask
for a `Window`, so a kernel on a remote server — a university machine with a GPU and
no screen — runs everything below.

This example is not run by CI, like every other example. Run it yourself.

It needs `pillow` to show the pictures and `ipywidgets` for the interactive cell:

    pip install bazalt pillow ipywidgets

**Two rules for a notebook**, and the second one is the one that bites:

1. Read your pixels inside the `with` block. A closed Context starts no new work.
2. Do not build a Context per redraw. Creating a device costs tens of milliseconds,
   so a slider that rebuilds one per event stops responding. Build it once and keep
   it, which is the second cell below.

In [ ]:
import struct

import numpy as np
from PIL import Image

import bazalt as bz

# Which GPU the kernel can see. On a login node this is often the answer to
# "why is nothing working" — no device means no Vulkan driver installed.
for device in bz.list_devices():
    print(device)

## A picture in a cell

`target.color[0].read()` gives a NumPy array, and everything that shows a NumPy array
shows this one. Bazalt has no display verb of its own on purpose: PIL, matplotlib and
imageio already answer that question, and they let you pick the layer and the mip.

One `with` block, one picture. This shape is right for a cell you run once.

In [ ]:
VERTEX = """
#version 450
layout(location = 0) out vec2 uv;
void main() {
    // One triangle covering the screen, with no vertex buffer. The order of the
    // two components is not free: it decides the winding, and bazalt culls back
    // faces by default. Swap them and the cell renders the clear colour.
    uv = vec2(gl_VertexIndex & 2, (gl_VertexIndex << 1) & 2);
    gl_Position = vec4(uv * 2.0 - 1.0, 0.0, 1.0);
}
"""

FRAGMENT = """
#version 450
layout(location = 0) in vec2 uv;
layout(location = 0) out vec4 color;
layout(push_constant) uniform Push { float scale; } push;
void main() {
    vec2 p = (uv - 0.5) * push.scale;
    float r = length(p);
    float rings = 0.5 + 0.5 * sin(40.0 * r - 6.0 * atan(p.y, p.x));
    color = vec4(rings * vec3(0.4, 0.7, 1.0) + 0.1, 1.0);
}
"""


def render_once(scale=1.0, size=512):
    """Build a Context, draw one frame, give back the pixels.

    Everything lives inside the `with`, including the read. A closed Context
    starts no new work, so target.color[0].read() outside the block raises
    StateError instead of handing back a black image.
    """
    with bz.Context() as ctx:
        target = ctx.create_render_target(size, size)
        pipeline = (ctx.graphics_pipeline()
                    .vertex_shader(ctx.compile_shader(source=VERTEX,
                                                      stage=bz.ShaderStage.VERTEX))
                    .fragment_shader(ctx.compile_shader(source=FRAGMENT,
                                                        stage=bz.ShaderStage.FRAGMENT))
                    .push_constant(4, bz.ShaderStage.FRAGMENT)
                    .build(target))

        cmd = ctx.create_command_buffer()
        cmd.begin()
        with cmd.rendering(target, clear_color=[0.02, 0.02, 0.05, 1.0]) as c:
            c.bind_pipeline(pipeline)
            c.push_constants(pipeline, 0, struct.pack("f", scale))
            c.draw(3)
        ctx.submit(cmd)

        return target.color[0].read()


Image.fromarray(render_once())

## What a notebook is actually for

Not one picture — a sweep. A notebook is a parameter editor with a shader in it, and
this is where it beats a windowed loop: the shader source is a cell, so you edit
`FRAGMENT` above, re-run these two cells, and drag the slider again.

`render_once` above is the wrong shape for that. It builds a Context, a pipeline and
a target per call, which costs tens of milliseconds each — a slider drag emits events
faster than that, so the queue never drains and the notebook stops responding.

So build all three once, and let a redraw be what a redraw is: record, submit, read.
The Context now outlives the cell, which is what `close()` is for — the last cell
releases it.

In [ ]:
SIZE = 384

# Kept in module scope on purpose: the widget callback below needs it between
# events, so it cannot live in a `with` block. The last cell closes it.
viewer = bz.Context()
viewer_target = viewer.create_render_target(SIZE, SIZE)
viewer_pipeline = (viewer.graphics_pipeline()
                   .vertex_shader(viewer.compile_shader(source=VERTEX,
                                                        stage=bz.ShaderStage.VERTEX))
                   .fragment_shader(viewer.compile_shader(source=FRAGMENT,
                                                          stage=bz.ShaderStage.FRAGMENT))
                   .push_constant(4, bz.ShaderStage.FRAGMENT)
                   .build(viewer_target))
viewer_cmd = viewer.create_command_buffer()


def draw(scale):
    """Re-record and submit. No device, no pipeline, no allocation."""
    viewer_cmd.begin()
    with viewer_cmd.rendering(viewer_target, clear_color=[0.02, 0.02, 0.05, 1.0]) as c:
        c.bind_pipeline(viewer_pipeline)
        c.push_constants(viewer_pipeline, 0, struct.pack("f", scale))
        c.draw(3)
    viewer.submit(viewer_cmd)
    return viewer_target.color[0].read()


print("one redraw:")
%timeit -n 5 -r 3 draw(1.0)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

out = widgets.Output()
slider = widgets.FloatSlider(value=1.0, min=0.2, max=4.0, step=0.1,
                             description="scale", continuous_update=False)


def on_change(change):
    with out:
        out.clear_output(wait=True)
        display(Image.fromarray(draw(change["new"])))


slider.observe(on_change, names="value")
display(slider, out)
on_change({"new": slider.value})

`continuous_update=False` on the slider means the callback runs when you let go of the
handle rather than on every pixel of the drag. Each redraw is a submit and a readback,
which blocks until the GPU is done, so this keeps a drag smooth. Turn it on once you
know how long your own shader takes.

## Compute, with no rendering at all

The other half a notebook is good for: the GPU as a calculator. A buffer in, a buffer
out, NumPy on both ends. This one is a single run, so it goes back to `with`.

In [ ]:
COMPUTE = """
#version 450
layout(local_size_x = 64) in;
layout(set = 0, binding = 0) buffer Data { float values[]; };
void main() {
    uint i = gl_GlobalInvocationID.x;
    if (i < values.length()) {
        values[i] = sqrt(values[i]);
    }
}
"""

data = np.arange(1024, dtype=np.float32)

with bz.Context() as ctx:
    pipeline = (ctx.compute_pipeline()
                .shader(ctx.compile_shader(source=COMPUTE, stage=bz.ShaderStage.COMPUTE))
                .storage_buffer(0)
                .build())
    buffer = ctx.create_buffer(data, bz.BufferType.STORAGE, bz.MemoryUsage.STATIC)
    pool = ctx.create_descriptor_pool()
    dset = pool.allocate_set(pipeline, set=0)
    dset.set_buffer(0, buffer)

    cmd = ctx.create_command_buffer()
    cmd.begin()
    cmd.bind_pipeline(pipeline).bind_descriptor_set(dset, pipeline, set=0).dispatch(1024 // 64)
    ctx.submit(cmd)

    result = buffer.read(np.float32)

print(result[:8])
print("matches numpy:", np.allclose(result, np.sqrt(data), atol=1e-5))

## Letting the viewer go

The Context from the interactive cell is still alive with its upload worker and its
hot-reload watcher. `close()` ends them at a point you pick, instead of whenever the
garbage collector notices.

It is idempotent, `ctx.closed` reports it, and the cached facts still answer
afterwards. What stops is starting new work.

In [ ]:
print("device:", viewer.device_name, "| headless:", viewer.headless)

viewer.close()
print("closed:", viewer.closed)
print("device after close:", viewer.device_name)

try:
    draw(1.0)
except bz.StateError as error:
    print("StateError:", error)